In [1]:
import numpy as np

In [2]:
def build_vocab(text) -> tuple[dict[str, int], int]:
    """Build a vocabulary from the input text and return a token table and its size.
    
    Vocab:
    A vocabulary is a mapping of unique words in the text to their corresponding indices. This is created using
    a pre-defined corpus of text, where each unique word is assigned a unique index.
    
    """
    words = text.split()
    token_table = {word: i for i, word in enumerate(set(words))}
    return token_table, len(token_table)

In [3]:
def tokenize(text: str, vocab: dict[str, int]) -> list[int]:
    """Tokenize the input text using the provided vocabulary.
    
    Tokenization:
    Tokenization is basically retrieving the ID or index of each word in a text input using an already built vocabulary.
    """
    tokens = [vocab[word] for word in text.split()]
    return tokens

In [4]:
def initialize_embeddings(vocab_size, embedding_dim=300):
    # Initialize a token embedding with random values
    E = np.random.rand(vocab_size, embedding_dim)
    return E

# test
vocab_size = 3
embedding_dim = 2

print(initialize_embeddings(vocab_size, embedding_dim))

[[0.85235143 0.93506138]
 [0.71730471 0.19305432]
 [0.73799841 0.06065846]]


In [5]:
# Example of embedding initialization
text = "hello world hello"

print("Input Text:", text)
vocab, vocab_size = build_vocab(text)
print("Vocabulary:", vocab)
print("Vocabulary Size:", vocab_size)
tokens = tokenize(text, vocab)
print("Tokens:", tokens)
embedding_table = initialize_embeddings(vocab_size, embedding_dim)
print("Embedding Table:\n", embedding_table)

Input Text: hello world hello
Vocabulary: {'hello': 0, 'world': 1}
Vocabulary Size: 2
Tokens: [0, 1, 0]
Embedding Table:
 [[0.76425084 0.20036349]
 [0.14912617 0.64175201]]


# Tokenization

## Byte Pair Encoding (BPE)
Byte Pair Encoding (BPE) is a data compression technique that iteratively replaces the most frequent pair of bytes in a sequence with a single, unused byte. This method is commonly used in natural language processing (NLP) for tokenization, where it helps to reduce the vocabulary size while maintaining the ability to represent a wide range of words and subwords. BPE is particularly effective in handling out-of-vocabulary words and can improve the performance of language models by allowing them to learn subword representations. The process involves creating a vocabulary of subword units based on the frequency of byte pairs,which can then be used for encoding and decoding text data.

### How BPE Works
1. **Initialization**: Start with a vocabulary that includes all individual characters (or bytes) in the text corpus.
2. **Counting Pairs**: Count the frequency of all adjacent byte pairs in the text.
3. **Merging**: Identify the most frequent byte pair and merge it into a new token. This new token is added to the vocabulary.
4. **Repeat**: Repeat the counting and merging process until a specified vocabulary size is reached or no more pairs can be merged.

In [6]:
# Byte Pair Encoding (BPE)

def byte_pair_encoding(text, vocab_size):
    # Initialize the vocabulary with individual characters
    vocab = {char: i for i, char in enumerate(set(text))}
    
    while len(vocab) < vocab_size:
        # Count frequency of adjacent pairs
        pairs = {}
        for i in range(len(text) - 1):
            pair = (text[i], text[i + 1])
            if pair in pairs:
                pairs[pair] += 1
            else:
                pairs[pair] = 1
        
        # Find the most frequent pair
        most_frequent_pair = max(pairs, key=pairs.get)
        
        # Merge the most frequent pair into a new token
        new_token = ''.join(most_frequent_pair)
        vocab[new_token] = len(vocab)
        
        # Replace occurrences of the pair in the text with the new token
        text = text.replace(''.join(most_frequent_pair), new_token)
    
    return vocab

# Embedding Initialization
Embedding initialization refers to the process of setting the initial values of the embedding vectors in a neural network. Proper initialization is crucial for the training of neural networks, as it can affect the convergence speed and overall performance of the model. Common methods for embedding initialization include:
1. **Random Initialization**: Embedding vectors are initialized with random values, often drawn from a uniform or normal distribution.
2. **Pre-trained Embeddings**: Using pre-trained embeddings (e.g., Word2Vec, GloVe) that have been trained on large corpora can provide a good starting point for the model, as they capture semantic relationships between words.
3. **Xavier/Glorot Initialization**: This simply samples a scalar from a uniform distribution where the standard deviation is a function of the number of input and output units in the weight tensor. The formula for Xavier initialization is:
$$\text{limit} = \sqrt{\frac{6}{\text{fan in} + \text{fan out}}}$$


**fan in**: The number of input units in the weight tensor.

**fan out**: The number of output units in the weight tensor.

In this notebook, we will use Xavier initialization.

In [7]:
# Xavier Initialization
def xavier_initialization(vocab_size, embedding_dim):
    limit = np.sqrt(6 / (vocab_size + embedding_dim))
    E = np.random.uniform(-limit, limit, (vocab_size, embedding_dim))
    return E

# Encoding and Decoding (Transformer)

Source: [Deep Dive into Transformers](https://www.datacamp.com/tutorial/how-transformers-work)

In the transformer architecture, the encoder uses an operation called "self-attention", which is *a way* to
